<a href="https://colab.research.google.com/github/peritopaulomax/PRNU/blob/master/PRNU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Implementação do algoritmo descrito em  *M. Goljan, T. Filler, and J. Fridrich. Large Scale Test of Sensor Fingerprint Camera Identification. In N.D. Memon and E.J. Delp and P.W. Wong and J. Dittmann, editors, Proc. of SPIE, Electronic Imaging, Media Forensics and Security XI, volume 7254, January 2009*.

O código abaixo foi desenvolvido tendo por base o disponibilizado em https://dde.binghamton.edu/download/camera_fingerprint/, que fora disponibilizado para fins **acadêmicos e de pesquisa**. Observe que o código lá disponibilizado está protegido pelas leis de direitos autorais dos Estados Unidos, sendo licenciado sob uma Licença Internacional Creative Commons Atribuição-NãoComercial 4.0, vedando o uso do código para fins comerciais sem a permissão expressa por escrito do titular dos direitos autorais. Para obter permissão, entre em contato com Scott Moser em smoser@binghamton.edu.

##### Desenvolvimento do código para uso em Jupiter Notebook com colaboração de:

##### Paulo Max Gil Innocencio Reis (paulo.pmgir@pf.gov.br)

##### Andrea Alves Guimarães Dresch

# **Importação das bibliotecas e funções necessárias. Bibliteca de funções deve estar carregada na pasta "*src*" após clonagem GIT**

In [ ]:
import os
def is_running_in_colab():
    return 'COLAB_GPU' in os.environ

iscolab=is_running_in_colab()

if iscolab:
    if os.path.exists('/content/PRNU'):
        !rm -rf {'/content/PRNU'}
    if not os.path.exists('/content/PRNU'):
        !git clone https://github.com/peritopaulomax/PRNU.git
    if not os.path.exists('/content/src'):
        !cp -R /content/PRNU/src /content/src
    if not os.path.exists('/content/padrao'):
        !mkdir /content/padrao
else:
    if not os.path.exists('./padrao'):
        !mkdir padrao
    if not os.path.exists('./src'):
        print("Instale a pasta 'src' a partir do repositorio em https://github.com/peritopaulomax/PRNU.git")

import src.Functions as Fu
import src.Filter as Ft
import src.getFingerprint as gF
import src.maindir as md
import src.extraUtils as eu
import numpy as np
import cv2 as cv
import pickle
import ipywidgets as widgets
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import LinearSegmentedColormap
import scipy as sp
from IPython.display import clear_output
from tqdm import tqdm




# **Utilize as celulas abaixo caso deseje calcular o PRNU de uma câmera alegada a partir de imagens padrão**



Definição dos caminhos da pasta contendo o material padrão (imagens coletadas a partir da câmera conhecida), As imagens padrão devem ser as mais homogêneas possivel (sem conteúdo em alta frequência), e submetidas a luminãncia elevada, sem saturação.

In [ ]:
if iscolab:
    caminho_pasta = '/content/padrao'
else:
    caminho_pasta ='./padrao'

lista = os.listdir(caminho_pasta)
padrao = [os.path.join(caminho_pasta, x) for x in lista]

Cálculo do padrão de PRNU da câmera que alegadamente é fonte da imagem questionada a partir das imagens padrão.

In [ ]:
RP,_,_ = gF.getFingerprint(padrao)
RP = Fu.rgb2gray1(RP)
sigmaRP = np.std(RP)
Fingerprint = Fu.WienerInDFT(RP, sigmaRP)
nome_arquivo_fingerprint = input("Digite o nome do arquivo para salvar PRNU: ")
if not nome_arquivo_fingerprint.endswith(".prnu"):
    nome_arquivo_fingerprint += ".prnu"

with open(nome_arquivo_fingerprint, 'wb') as arquivo:
    pickle.dump(Fingerprint, arquivo)

# **Utilize a célula abaixo caso deseje carregar um arquivo PRNU calculado anteriormente**
    Modifique o nome do arquivo de acordo com o seu em específico.

  * Não é necessario recarregar se tiver acabado de gerar o arquivo PRNU no passo anterior


In [ ]:
if iscolab:
    with open('/content/camera1.prnu', 'rb') as arquivo:
        Fingerprint = pickle.load(arquivo)
else:
    with open('./Padrao D70.prnu', 'rb') as arquivo:
        Fingerprint = pickle.load(arquivo)

# **Extração do resíduo de ruído da imagem questionada para comparação com o padrão PRNU da câmera alegada**

In [ ]:
if iscolab:
    questionado = '/content/10.JPG'
else:
    questionado = '4-040-085_366_258.jpg'

Noisex = Ft.NoiseExtractFromImage(questionado, sigma=2.)
Noisex = Fu.WienerInDFT(Noisex, np.std(Noisex))


# **Escolha do modo de comparação entre padrão e questionado**

In [ ]:
def tratar_selecao(opcao):
  global selecao_radiobutton
  selecao_radiobutton = opcao
  print("Opção selecionada:", opcao)
# Criação da combobox com as opções
opcoes = ['Completa', 'Recortada', 'Recortada e Redimensionada']
radiobutton = widgets.RadioButtons(options=opcoes)

# Exibição da combobox
#display(radiobutton)
widgets.interact(tratar_selecao, opcao=radiobutton)


# **Comparação entre questionado e padrão**

In [ ]:
import numpy as np
import cv2 as cv
from joblib import Parallel, delayed
from tqdm import tqdm
passo= 0.0005
esc_max=0.6

def process_single_rinv(rinv, Noisex, Ix, Fingerprint, rmin):
    #print(f'Razões máxima, atual e mínima:10.0000, {1/rinv:.4f}, {rmin:.4f}')
    Nsized = Noisex#cv.resize(Noisex, None, fx=rinv, fy=rinv, interpolation=cv.INTER_LANCZOS4)
    Ixsized = Ix#cv.resize(Ix, None, fx=rinv, fy=rinv, interpolation=cv.INTER_LANCZOS4)
    Fingerprint = cv.resize(Fingerprint, None, fx=1/rinv, fy=1/rinv, interpolation=cv.INTER_LINEAR)
    A=Nsized
    F=Fingerprint
    if Fingerprint.shape[0] > Nsized.shape[0] or Fingerprint.shape[1] > Nsized.shape[1] :
        A = np.pad(Nsized, [(0, abs(2*Fingerprint.shape[0]-1-Nsized.shape[0])), (0, abs(2*Fingerprint.shape[1]-1-Nsized.shape[1]))])
        F = np.pad(Fingerprint, [(0, abs(Fingerprint.shape[0]-1)), (0, abs(Fingerprint.shape[1]-1))])
    C = Fu.crosscorr(A, F)
    if min([Fingerprint.shape[0] - Nsized.shape[0], Fingerprint.shape[1] - Nsized.shape[1]]) >= 0:
        det, det0 = md.PCE(C, [Fingerprint.shape[0] - Nsized.shape[0], Fingerprint.shape[1] - Nsized.shape[1]])
        loc = det['PeakLocation']
        C = Fu.crosscorr(Nsized, Fingerprint[loc[0]:loc[0]+Ixsized.shape[0], loc[1]:loc[1]+Ixsized.shape[1]])
        det, det0 = md.PCE(C)
        return det, 1/rinv, loc, C
    return None, None, None, None

def parallel_processing(Noisex, Ix, Fingerprint, rmin):
    rinv_range = np.arange(1/esc_max, 1/rmin, passo)

    results = Parallel(n_jobs=-1)(delayed(process_single_rinv)(rinv, Noisex, Ix, Fingerprint, rmin) for rinv in tqdm(rinv_range, desc="Processando", unit="rinv"))
    
    det_max = {'PCE': -np.inf}
    rT = None
    locT = None
    pcepce = []
    
    for det, r, loc, C in results:
        if det is not None:
            pcepce.append(det['PCE'])
            if det['PCE'] > det_max['PCE']:
                det_max = det
                rT = r
                locT = loc
                C_Max = C
    
    return det_max, rT, locT, pcepce, C



In [ ]:
from scipy.signal import correlate2d



Ix = cv.cvtColor(cv.imread(questionado),# image in BGR format
                 cv.COLOR_BGR2GRAY)

if selecao_radiobutton == "Recortada":
    A=Noisex
    F=Fingerprint
    if Fingerprint.shape[0] > A.shape[0] or Fingerprint.shape[1] > A.shape[1]:
        A = np.pad(A, [(0, abs(2*Fingerprint.shape[0]-1-A.shape[0])), (0, abs(2*Fingerprint.shape[1]-1-A.shape[1]))])
        F = np.pad(F, [(0, abs(Fingerprint.shape[0]-1)), (0, abs(Fingerprint.shape[1]-1))])
    C = Fu.crosscorr(A, F)
    det, det0 = md.PCE(C, [Fingerprint.shape[0]-Noisex.shape[0], Fingerprint.shape[1]-Noisex.shape[1]],11)
    loc = det['PeakLocation']
    C = Fu.crosscorr(Noisex, Fingerprint[loc[0]:loc[0]+Ix.shape[0], loc[1]:loc[1]+Ix.shape[1]])
    det, det0 = md.PCE(C)
    det['PeakLocation']=loc


elif selecao_radiobutton == "Completa":
    aux = Fingerprint
    if Ix.shape[0] > Ix.shape[1]:
        Ix = np.rot90(Ix,-1)
        Noisex = np.rot90(Noisex,-1)

    if Fingerprint.shape[1] > Ix.shape[1]:
        Fingerprint = cv.resize(Fingerprint, (Ix.shape[1], Ix.shape[0]), interpolation=cv.INTER_LINEAR)

    if np.size(Ix) != np.size(Fingerprint):
        raise ValueError("PRNU e Questionada com dimensões diferentes. Verifique outro método")

    C= Fu.crosscorr(Noisex,Ix*Fingerprint)
    det, det0 = md.PCE(C)
    Fingerprint = aux
elif selecao_radiobutton == "Recortada e Redimensionada":
    det_max = {"PCE": 0}
    rT = 1
    locT = [0, 0]
    rmin = max([Noisex.shape[0] / Fingerprint.shape[0], Noisex.shape[1]/ Fingerprint.shape[1]])
    r = 1
    i = 0
    pcepce=[]
    det_max, rT, locT, pcepce, C= parallel_processing(Noisex, Ix, Fingerprint, rmin)
    det=det_max
    det['PeakLocation']=locT
    print(f'A razão de reeescalnamento encontrada foi:{rT:5f}')
    print(f'A imagem foi recortada em: X={locT[0]} e Y={locT[1]}')
    plt.plot(1/np.arange(1/esc_max, 1/rmin, passo),pcepce)
    plt.xlabel('Razão de reescalonamento')
    plt.ylabel('PCE')
    plt.title('Detecção de reescalonamento')
    plt.show()

In [ ]:
Noisex.shape[0]

In [ ]:
for key in det.keys(): print("{0}: {1}".format(key, det[key]))
logpce=np.log10(det['PCE'])
print(f'Log(PCE):{logpce}')


In [ ]:
Z =np.fft.fftshift(C)
X = np.arange(Z.shape[0])
Y = np.arange(Z.shape[1])
X, Y = np.meshgrid(Y,X)
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_facecolor((0.7, 0.7, 0.7))
cmap = LinearSegmentedColormap.from_list(
   "mycmap", [(0, "blue"),(0.4, "white"), (0.7, "green"),(1, "red")]
)
ax.contour3D(X, Y, Z, 200, cmap=cmap)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()